# Week 8, Day 2: Derivative rules and the nonlinearities of neurons and networks

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w008/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Code the derivatives

Write `dsigmoid`, `dtanh`, `dsoftplus` and `drelu`, each working on whole arrays. Use $\sigma' = \sigma(1 - \sigma)$, $\tanh' = 1 - \tanh^2$ and softplus$' = \sigma$. Check them all against central differences, then print each derivative's largest value over the grid.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def sigmoid(x): return 1 / (1 + np.exp(-x))
def softplus(x): return np.log1p(np.exp(x))
def relu(x): return np.maximum(0, x)

def dsigmoid(x):
    return ...
def dtanh(x):
    return ...
def dsoftplus(x):
    return ...
def drelu(x):
    return ...

x = np.linspace(-6, 6, 1201)
for name, d in [("sigmoid", dsigmoid), ("tanh", dtanh), ("softplus", dsoftplus), ("relu", drelu)]:
    print(f"{name:9s} largest slope {d(x).max():.3f}")

In [ ]:
import numpy as np
xs = np.linspace(-4, 4, 17) + 0.013   # avoid the ReLU kink at exactly 0
def num(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

def test_sigmoid():
    "dsigmoid matches a numerical derivative"
    assert np.allclose(dsigmoid(xs), num(sigmoid, xs), atol=1e-7)

def test_tanh():
    "dtanh matches a numerical derivative"
    assert np.allclose(dtanh(xs), num(np.tanh, xs), atol=1e-7)

def test_softplus():
    "dsoftplus matches a numerical derivative"
    assert np.allclose(dsoftplus(xs), num(softplus, xs), atol=1e-7)

def test_relu():
    "drelu is 1 for positive inputs and 0 for negative ones"
    assert np.array_equal(drelu(xs), (xs > 0).astype(float))

test_sigmoid()
test_tanh()
test_softplus()
test_relu()
print("All checks passed")

### Exercise: Where is a neuron most sensitive to contrast?

Differentiate the Naka-Rushton function with the quotient rule (numerator $f = R_{\max}c^n$, denominator $g = c^n + c_{50}^n$) and write `dnaka(c)`. Then find `c_best`, the contrast where the slope is largest, on the grid `cs`. Is the neuron most sensitive at $c_{50}$?

In [ ]:
Rmax, c50, n = 50.0, 0.2, 2.5
def naka(c):
    return Rmax * c ** n / (c ** n + c50 ** n)

def dnaka(c):
    f, g = Rmax * c ** n, c ** n + c50 ** n
    df, dg = ..., ...
    return ...

cs = np.linspace(0.001, 1, 100000)
c_best = ...
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(cs, naka(cs)); ax[0].axvline(c50, ls="--", color="0.5"); ax[0].set_xlabel("contrast"); ax[0].set_ylabel("rate (Hz)")
ax[1].plot(cs, dnaka(cs)); ax[1].axvline(c_best, ls="--", color="C1"); ax[1].set_xlabel("contrast"); ax[1].set_ylabel("slope (Hz per unit contrast)")
plt.tight_layout(); plt.show()
print("most sensitive at contrast", round(c_best, 4), " c50 =", c50)

In [ ]:
import numpy as np
def test_deriv():
    "dnaka matches a numerical derivative"
    for c in [0.05, 0.15, 0.3, 0.7]:
        assert np.isclose(dnaka(c), (naka(c + 1e-6) - naka(c - 1e-6)) / 2e-6, rtol=1e-5)

def test_peak():
    "c_best is where the slope peaks, below c50 for n = 2.5"
    assert np.isclose(c_best, cs[np.argmax(dnaka(cs))])
    assert 0.1 < c_best < c50

test_deriv()
test_peak()
print("All checks passed")